In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.3 The Large-N Limit: Stirling, the CLT, and Sharp Macrostates

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.3",
    title="The Large-N Limit: Stirling, the CLT, and Sharp Macrostates",
    blurb="What happens when the numbers get enormous. Factorials overflow and "
    "force us into log space, where Stirling's approximation turns a count "
    "into an entropy; sums of anything become Gaussian; and fluctuations "
    "shrink as 1/√N — which is why a mole of gas has a temperature at all.",
    difficulty="advanced",
    estimate="140–180 min",
)

## Notebook overview

This is the last notebook of Volume V's mathematical arsenal, and the one where the
arsenal turns into physics. Counting ([§5.1](counting.ipynb)) and probability ([§5.2](probability.ipynb)) gave us the tools; here we
take them to the limit that statistical mechanics actually lives in, $N\sim10^{23}$, and three
things happen that together make thermodynamics possible.

First, a computational wall. The factorials that count microstates grow so fast that
$1000!$ overflows a 64-bit float outright — a direct return of the floating-point limits of
[§0.1](../00-foundations/floating-point.ipynb). The escape is to stop computing the count and compute its **logarithm**, with
`scipy.special.gammaln`, and this is not a mere trick: statistical mechanics is written in log
space because that is the only space its numbers fit in. **Stirling's approximation** then
turns $\ln N!$ into a clean closed form, the analytic handle that converts a factorial count
into an entropy.

Second, a universal law. The **central limit theorem** says that the sum of many independent
random variables, *whatever* their individual distribution, approaches a Gaussian. We watch a
perfectly flat distribution become a bell curve in about thirty steps, and we see the binomial
of a two-state system do the same. The Gaussian is not one distribution among many; it is the
large-$N$ attractor of nearly all of them.

Third, and deepest, **sharpness**. For $N$ independent contributions the sum has mean
proportional to $N$ but standard deviation only proportional to $\sqrt N$, so the *relative*
fluctuation falls as $1/\sqrt N$. At Avogadro's number that is about one part in $10^{12}$:
the spread is so negligible that a thermodynamic variable — an energy, a pressure, a
temperature — has an effectively exact value. Macroscopic determinism, the whole edifice of
thermodynamics, is the law of large numbers in disguise. We close by assembling these pieces
into the **entropy of mixing**, $S=k\ln\Omega$, the first formula of statistical mechanics and
the hand-off to the physics that begins at [§5.4](microstates-entropy-temperature.ipynb).

We reuse the distribution-bar plots and Monte Carlo machinery of [§5.1](counting.ipynb)–[§5.2](probability.ipynb), verify our
hand-built results against `scipy` where useful, and keep every $N$ and distribution stated
explicitly.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: factorials overflowing while `gammaln` stays finite; Stirling matching
> $\ln N!$; the sample mean converging; a sum of uniforms with mean $N/2$ and standard
> deviation $\sqrt{N/12}$ going Gaussian; the binomial→Gaussian limit; $\sigma/\mu\propto1/
> \sqrt N$; the mixing entropy approaching $-\sum x_i\ln x_i$. A ✓ is strong evidence; a ✗ is
> a prompt to *locate the discrepancy*, not a verdict.
>
> **Scope.** The large-$N$ limit that bridges the math arsenal to the physics; the physics
> proper — microstates, the Boltzmann distribution, temperature — begins at [§5.4](microstates-entropy-temperature.ipynb). See
> Schroeder, *Thermal Physics*; Kardar, *Statistical Physics of Particles*; Feller; and
> [§0.1](../00-foundations/floating-point.ipynb) (floating point), [§5.1](counting.ipynb) (counting), [§5.2](probability.ipynb) (probability).

## Theory in brief

### The computational wall: factorials overflow

A microstate count is a factorial, and factorials grow faster than any exponential. $1000!$
has $2568$ digits and overflows a $64$-bit float ($\max\approx1.8\times10^{308}$),

```{math}
:label: eq-overflow
\ln N! = \ln\Gamma(N+1) \quad\text{(computed without overflow by }\texttt{scipy.special.gammaln}\text{)} .
```

The remedy, a direct callback to [§0.1](../00-foundations/floating-point.ipynb), is to work with $\ln N!$ throughout: statistical
mechanics lives in log space.

### Stirling's approximation

The logarithm has a beautifully simple large-$N$ form,

```{math}
:label: eq-stirling
\ln N! \approx N\ln N - N \quad\text{(leading)}, \qquad \ln N! \approx N\ln N - N + \tfrac12\ln(2\pi N) \quad\text{(corrected)} .
```

The leading form is already good to $\sim0.9\%$ at $N=100$; the corrected one to $\sim2\times
10^{-4}\%$. Stirling is the analytic handle that turns a factorial count into a closed-form
expression, and the gateway to entropy. We verify the approximation numerically rather than
derive it; Reif, *Fundamentals of Statistical and Thermal Physics*, App. A.6, carries the
derivation out in full.

### The law of large numbers and the central limit theorem

Two limit theorems govern sums of $N$ independent, identically distributed variables. The
**law of large numbers** says the sample mean converges to the true mean,

```{math}
:label: eq-lln
\bar X_N=\frac1N\sum_{i=1}^N X_i \xrightarrow{N\to\infty} \langle X\rangle ,
```

and the **central limit theorem** says the fluctuations around it are Gaussian, whatever the
distribution of each $X_i$,

```{math}
:label: eq-clt
\sum_{i=1}^N X_i \approx \mathcal N\!\big(N\mu,\ N\sigma^2\big) \quad\text{for large }N .
```

The Gaussian is the universal large-$N$ attractor; the binomial→Gaussian case is the discrete
instance (de Moivre–Laplace). We watch both theorems act rather than prove them; Feller,
*An Introduction to Probability Theory*, supplies the proofs in full.

### Fluctuations and the sharpness of macrostates

Because the sum has mean $\propto N$ and standard deviation $\propto\sqrt N$, the relative
fluctuation shrinks,

```{math}
:label: eq-fluctuations
\frac{\sigma}{\mu}\propto\frac{1}{\sqrt N} \;\xrightarrow{N=N_A}\; \sim10^{-12} .
```

At macroscopic $N$ the relative spread is utterly negligible, so a thermodynamic variable has
an effectively **sharp** value. This is why thermodynamics is deterministic though built on
chance.

### The bridge to physics: entropy

Stirling turns the log of a multiplicity into an extensive quantity. For $N$ particles split
into species with fractions $x_i$,

```{math}
:label: eq-entropy
\ln\Omega=\ln\frac{N!}{\prod_i N_i!} \xrightarrow{\text{Stirling}} -N\sum_i x_i\ln x_i, \qquad S=k\ln\Omega .
```

This is the **entropy of mixing**, and $S=k\ln\Omega$ is the first formula of statistical
mechanics: entropy is the logarithm of a count, computable only in log space, extensive and
sharp only because $N$ is large.

## Setup

In [ ]:
from math import comb, factorial, log

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy import stats
from scipy.special import gammaln

from ecp import combinatorics as cb
from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
from scipy.constants import N_A as N_AVOGADRO


def stirling_leading(n):
    """Stirling's leading approximation to ln n!: n ln n − n (eq-stirling).

    Parameters
    ----------
    n : float or numpy.ndarray
        The argument (treated as continuous).

    Returns
    -------
    float or numpy.ndarray
        The leading-order ln n!.
    """
    return n * np.log(n) - n


def stirling_corrected(n):
    """Stirling's approximation with the first correction, n ln n − n + (1/2) ln(2πn) (eq-stirling).

    Parameters
    ----------
    n : float or numpy.ndarray
        The argument.

    Returns
    -------
    float or numpy.ndarray
        The corrected ln n!, accurate to a few parts in 10^6 by n ~ 10^3.
    """
    return n * np.log(n) - n + 0.5 * np.log(2.0 * np.pi * n)


def log_multiplicity(counts):
    """The log-multiplicity ln Ω = ln(N!/Π_i N_i!) via ``scipy.special.gammaln`` (eq-entropy).

    Computed in log space (never the multiplicity itself) so it never overflows, with
    ln N! = ``gammaln(N+1)``.

    Parameters
    ----------
    counts : array_like
        The species counts N_i (summing to N).

    Returns
    -------
    float
        The log-multiplicity ln Ω.
    """
    counts = np.asarray(counts, dtype=float)
    return float(gammaln(counts.sum() + 1.0) - np.sum(gammaln(counts + 1.0)))

## Exercise 1 — The factorial wall and log space

Statistical mechanics counts microstates, and microstate counts are factorials, and factorials
are where ordinary arithmetic breaks. We met the limits of floating point in [§0.1](../00-foundations/floating-point.ipynb); here
they bite hard. The number $1000!$ has $2568$ digits, far past the $\sim308$-digit ceiling of a
$64$-bit float, so asking for it as a float overflows to infinity. The escape is to never
compute the count itself but its **logarithm**, $\ln N!=\ln\Gamma(N+1)$, which
`scipy.special.gammaln` evaluates directly and without overflow {eq}`eq-overflow`. From here
on we live in log space, because the numbers of statistical mechanics simply do not fit
anywhere else.

**This exercise (worked).** Show that converting $1000!$ to a float overflows (raising
`OverflowError`), while `scipy.special.gammaln(1001)` returns a finite $\ln(1000!)$; confirm
`gammaln(n+1)` equals $\ln n!$ for small $n$ where the exact factorial is still computable; and
tabulate $\ln N!$ for growing $N$. *Forward-pointer:* every entropy in this volume is a
`gammaln`, never a factorial.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    overflowed and np.isfinite(gammaln(1001)),
    "factorials overflow a float; their logarithms via gammaln do not",
)
validate.close(
    gammaln(np.arange(1, 11) + 1),
    np.array([log(factorial(n)) for n in range(1, 11)]),
    "gammaln(n+1) computes ln(n!)",
    rtol=1e-9,
)

## Exercise 2 — Stirling's approximation

Working in log space proves its worth immediately, because $\ln N!$ has a remarkably simple large-$N$
form. **Stirling's approximation** is $\ln N!\approx N\ln N-N$, with a first correction
$+\tfrac12\ln(2\pi N)$ {eq}`eq-stirling`. The leading form already tracks the exact value to
under a percent at $N=100$, and the corrected form to a few parts in a million
({numref}`fig-ln-stirling`). This is the analytic lever of the whole subject: it replaces an
intractable factorial with elementary functions of $N$, and it is what lets a log-multiplicity
collapse into a closed-form entropy in Exercise 7.

**This exercise (worked).** Compare both Stirling forms (the `stirling_leading` and
`stirling_corrected` helpers) to the exact $\ln N!$ from `scipy.special.gammaln` across $N$;
confirm the corrected form matches to better than $10^{-4}$ relative at $N=1000$, and that both
relative errors fall with $N$. The figure plots both errors against $N$ on log–log axes.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    stirling_corrected(1000.0),
    gammaln(1001.0),
    "Stirling's approximation (with the ½ln(2πN) term) matches ln(N!) at large N",
    rtol=1e-4,
)
validate.check(
    err_leading[-1] < err_leading[0] and err_corrected[-1] < err_corrected[0],
    "both Stirling relative errors shrink with N",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The law of large numbers

Before the subtler limit theorem, the simple one. The **law of large numbers** says that the
average of many independent draws converges to the true mean {eq}`eq-lln`. Roll a fair die
repeatedly and keep a running average: it wanders at first, then settles inexorably onto $3.5$
({numref}`fig-ln-lln`). This is the same convergence that made Monte Carlo work in [§5.2](probability.ipynb), now
named, and it is the first reason large systems behave predictably: averaged over enough
constituents, chance washes out.

**This exercise (worked).** Draw a long sequence of die rolls with `numpy.random.default_rng`,
form the running sample mean with `numpy.cumsum`, and confirm it converges to the true mean
$3.5$. The figure shows the running mean settling, with the characteristic $1/\sqrt N$ envelope.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    running_mean[-1],
    3.5,
    "the sample mean converges to the true mean (law of large numbers)",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The central limit theorem from a flat distribution

Here is the striking one. The **central limit theorem** says the sum of many independent
variables tends to a Gaussian *no matter what distribution each one has* {eq}`eq-clt`. To see
how universal that is, start from the least Gaussian distribution imaginable: the flat
uniform$[0,1]$. The sum of one is flat; the sum of two is a triangle; and by about thirty the
sum is a bell curve indistinguishable from a Gaussian by eye ({numref}`fig-ln-clt`). The sum of
$N$ uniforms has mean $N/2$ and variance $N/12$, and its skewness and excess kurtosis — the
numerical fingerprints of non-Gaussianity — march to zero.

**This exercise (worked).** With `numpy.random.default_rng`, draw many sums of $N$ uniform
variables; confirm the sample mean and standard deviation match $N/2$ and $\sqrt{N/12}$ (via
`numpy.mean`/`numpy.std`), and that the skewness and excess kurtosis (`scipy.stats.skew`,
`scipy.stats.kurtosis`, for checking only) fall toward zero as $N$ grows. The animation morphs
the standardized distribution from flat toward the Gaussian.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    np.array([sum_mean, sum_std]),
    np.array([N_clt / 2, np.sqrt(N_clt / 12)]),
    "the sum of N uniforms has mean N/2 and standard deviation √(N/12)",
    rtol=1e-2,
)
validate.check(
    abs(stats.kurtosis(rng.uniform(0, 1, size=(80_000, 30)).sum(axis=1))) < 0.1,
    "the sum approaches a Gaussian (excess kurtosis → 0) regardless of the flat base distribution",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The binomial becomes Gaussian: de Moivre–Laplace

The two-state system of [§5.1](counting.ipynb) and [§5.2](probability.ipynb) — coins, spins, the paramagnet — gives the discrete instance
of the same limit. The **de Moivre–Laplace** theorem says the binomial$(N,\tfrac12)$ approaches
a Gaussian of matching mean $N/2$ and variance $N/4$ as $N$ grows {eq}`eq-clt`. By $N=100$ the
two agree to a couple parts in $10^4$ everywhere ({numref}`fig-ln-binom`). This is why the
multiplicity of a large paramagnet — the number of ways to have $k$ spins up — is a sharp
Gaussian peak about $k=N/2$, the fact we draw on for the sharpness of macrostates and, at [§5.4](microstates-entropy-temperature.ipynb),
for its energy distribution.

**This exercise (worked).** Build the binomial$(100,\tfrac12)$ pmf with `math.comb`, overlay
the matched Gaussian, and confirm the maximum absolute difference is below $10^{-3}$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    binom_pmf,
    matched_gauss,
    "the binomial converges to a Gaussian (de Moivre–Laplace)",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Fluctuations shrink as 1/√N

This is the payoff the whole arsenal was built for. Take $N$ independent contributions to some
extensive quantity. Their sum has mean proportional to $N$ and, because variances add ([§5.2](probability.ipynb)),
standard deviation proportional to $\sqrt N$. The **relative** fluctuation is therefore the
ratio, $\sigma/\mu\propto1/\sqrt N$ {eq}`eq-fluctuations`, and it shrinks without bound as the
system grows ({numref}`fig-ln-fluct`). At $N=100$ it is about $10\%$; at $N=10^4$, about $1\%$;
at Avogadro's number it is about $1.3\times10^{-12}$, one part in a trillion. That is the deepest
fact in this notebook: at macroscopic $N$ the relative spread of a thermodynamic variable is so
small that the variable is, for all purposes, **sharp**. A mole of gas has a definite energy,
pressure, and temperature not because chance has been banished but because the law of large
numbers has made it invisible.

**This exercise (worked).** Tabulate $\sigma/\mu=1/\sqrt N$ for $N$ from $100$ to Avogadro's
number, confirm it reaches $\sim10^{-12}$ there, and verify the $1/\sqrt N$ scaling directly by
simulating sums of die rolls with `numpy.random.default_rng` and measuring $\sigma/\mu$ with
`numpy.std`/`numpy.mean`.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    rel_sim * np.sqrt(Ns_sim),
    np.full(4, rel_sim[0] * np.sqrt(Ns_sim[0])),
    "the relative fluctuation scales as 1/√N",
    rtol=5e-2,
)
validate.close(
    1.0 / np.sqrt(N_AVOGADRO),
    1.3e-12,
    "at Avogadro's number the relative fluctuation is ~10⁻¹²",
    rtol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — From a count to an entropy: the entropy of mixing

Now the bridge, where the math arsenal becomes physics. The number of ways to arrange $N$
particles of two species, $N_A$ of one and $N_B$ of the other, is the multiplicity $\Omega=
N!/(N_A!\,N_B!)$ — a single binomial coefficient, far too large to compute directly but
perfectly tractable in log space {eq}`eq-entropy`. Feed $\ln\Omega$ through Stirling and the
factorials collapse into something extensive and physical,

$$\frac{\ln\Omega}{N}\xrightarrow{\text{Stirling}}-\big(x_A\ln x_A+x_B\ln x_B\big),$$

the **entropy of mixing per particle**, with $x_i=N_i/N$. Multiply by Boltzmann's constant and
this is $S=k\ln\Omega$, the first formula of statistical mechanics. Everything we built points
here: entropy is the logarithm of a count ([§5.1](counting.ipynb)), made finite by working in log space (this
notebook, and [§0.1](../00-foundations/floating-point.ipynb)), and extensive only because Stirling's sub-leading terms vanish per particle
as $N$ grows ({numref}`fig-ln-entropy`).

**This exercise (worked).** For a mixture with $x_A=0.3$, compute $\ln\Omega$ exactly with
`scipy.special.gammaln` (the `log_multiplicity` helper) and via Stirling, confirm they agree;
and show the per-particle entropy $\ln\Omega/N$ converges to $-\sum_i x_i\ln x_i$ as $N$ grows.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    ln_omega_stirling_big,
    ln_omega_exact_big,
    "Stirling reproduces the log-multiplicity of the mixed state",
    rtol=1e-3,
)
validate.close(
    ln_omega_exact_big / N_big,
    mixing_entropy_target,
    "the per-particle mixing entropy approaches −Σ xᵢ ln xᵢ",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The most probable macrostate dominates

We can now see, in one picture, why equilibrium is what it is. For $N$ coins (or spins), the
number of microstates with $k$ heads is the multiplicity $\Omega(k)=\binom{N}{k}$, sharply
peaked at $k=N/2$ {eq}`eq-fluctuations`. As $N$ grows the peak does not merely stay put — it
*narrows*, with a relative width $\propto1/\sqrt N$, so an ever-larger fraction of all
microstates piles up within a hair of $k=N/2$ ({numref}`fig-ln-peak`). At macroscopic $N$ the
states even slightly away from the peak are so vastly outnumbered that they are never observed.
This is the whole content of equilibrium: the macrostate we call equilibrium is simply the one
realised by the overwhelming majority of microstates, and at large $N$ it is effectively the
*only* one. The second law is this counting fact at scale.

**This exercise (student).** Using `scipy.special.gammaln` to stay in log space, compute the
normalized multiplicity $\Omega(k)/\Omega_{\max}$ for several $N$, confirm it peaks at $k=N/2$
and narrows as $1/\sqrt N$, and that the fraction of microstates within a fixed *relative*
window of the peak approaches $1$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    widths[0] > widths[1] > widths[2]
    and np.allclose(
        np.array(widths) * np.sqrt([100, 1000, 10000]), widths[0] * 10.0, rtol=1e-2
    ),
    "the most probable macrostate (k=N/2) dominates: the multiplicity peak narrows as 1/√N",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — The arsenal becomes physics

Stand back and see the whole arc. We set out three notebooks ago simply to **count** — socks,
poker hands, balls in boxes ([§5.1](counting.ipynb)). We turned counts into **probabilities** and defined the
expectation and variance that quantum mechanics will reuse verbatim ([§5.2](probability.ipynb)). And here, in the
**large-$N$ limit**, the mathematics has become physics. Factorials forced us into log space,
where Stirling turns a count into an entropy; the central limit theorem explained why the
Gaussian is everywhere; and the $1/\sqrt N$ collapse of fluctuations revealed why a system of
$10^{23}$ parts has sharp, deterministic macroscopic properties at all. The first formula of
statistical mechanics, $S=k\ln\Omega$, fell out of a binomial coefficient and Stirling's
approximation. We set out to count, and we have arrived at entropy.

**This exercise.** Confirm the synthesis in one line: that the three pillars agree where they
must meet — the entropy of an evenly mixed system, $S/N=\ln\Omega/N$ at $x_A=x_B=\tfrac12$,
equals $\ln 2$, the same $\ln 2$ that is the per-spin entropy of the two-state system and the
information in one bit. Counting, probability, and the large-$N$ limit are one subject.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    ln_omega_even,
    np.log(2),
    "the even-mixing entropy per particle is ln 2 — counting, probability, and large-N meet",
    rtol=1e-3,
)

## Notebook summary

This notebook closes **Volume V's mathematical arsenal** by taking counting and probability to
the limit statistical mechanics lives in, and handing off to the physics.

- **The factorial wall** {eq}`eq-overflow`: $1000!$ overflows a float, so we work with $\ln N!=$
  `scipy.special.gammaln`$(N+1)$ — statistical mechanics lives in log space (the [§0.1](../00-foundations/floating-point.ipynb) callback).
- **Stirling** {eq}`eq-stirling`: $\ln N!\approx N\ln N-N$ (corrected $+\tfrac12\ln 2\pi N$),
  relative error $\sim0.9\%$ at $N=100$ and a few parts in $10^{6}$ corrected — the analytic
  handle that turns a count into a closed form.
- **The limit theorems** {eq}`eq-lln`, {eq}`eq-clt`: the sample mean converges (law of large
  numbers); the sum of $N$ uniforms goes Gaussian by $N\approx30$ (skew, kurtosis $\to0$), and
  the binomial$(100,\tfrac12)$ matches a Gaussian to $2\times10^{-4}$ (de Moivre–Laplace).
- **Sharpness** {eq}`eq-fluctuations`: $\sigma/\mu=1/\sqrt N$, about $1.3\times10^{-12}$ at
  Avogadro's number — why a mole of gas has a sharp temperature and thermodynamics is
  deterministic.
- **The bridge to physics** {eq}`eq-entropy`: $\ln\Omega/N\to-\sum_i x_i\ln x_i$ via Stirling,
  the entropy of mixing; $S=k\ln\Omega$ is the first formula of statistical mechanics; and the
  multiplicity peaks so sharply at $k=N/2$ that the most probable macrostate is the only one
  observed — the second law as a counting fact.

We set out to count, and we arrived at entropy. The physics starts at [§5.4](microstates-entropy-temperature.ipynb): microstates, the
Boltzmann distribution, and temperature and entropy emerging from the counting built here.

## Outlook

- **Statistical mechanics proper ([§5.4](microstates-entropy-temperature.ipynb) onward).** Microstates, ensembles, the Boltzmann distribution,
  and temperature and entropy from counting — the physics the arsenal was built for.
- **Monte Carlo and molecular dynamics.** The computational engines of the volume, building on
  the simulation spine of [§5.1](counting.ipynb)–§5.3.
- **The thermodynamic limit and phase transitions.** Where the $1/\sqrt N$ sharpness can break
  down — at a critical point fluctuations grow and span all scales (a pointer).
- **Quantum statistics (Volume VII).** The same counting of [§5.1](counting.ipynb), now for indistinguishable
  particles, giving the Fermi–Dirac and Bose–Einstein distributions.
- **Cross-reference** [§0.1](../00-foundations/floating-point.ipynb) (floating point), [§5.1](counting.ipynb) (counting), and [§5.2](probability.ipynb) (probability).

In [ ]:
from ecp.style import footer

footer()